In [178]:
import pandas as pd

In [179]:
df_order = pd.read_csv('../dataset1/orders.csv')
df_order.info()
print(df_order.head(5))

<class 'pandas.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   order_id        646945 non-null  int64
 1   order_date      646945 non-null  str  
 2   customer_id     646945 non-null  int64
 3   zip             646945 non-null  int64
 4   order_status    646945 non-null  str  
 5   payment_method  646945 non-null  str  
 6   device_type     646945 non-null  str  
 7   order_source    646945 non-null  str  
dtypes: int64(3), str(5)
memory usage: 39.5 MB
   order_id  order_date  customer_id   zip order_status payment_method  \
0         1  2012-07-04        58578  1109    delivered    credit_card   
1         2  2012-07-04        58621  1330     returned            cod   
2         3  2012-07-04        58811  1473    delivered    credit_card   
3         4  2012-07-04        59453  2360    delivered    credit_card   
4         6  2012-07-06        57821  2886  

In [180]:
df_order = df_order[['order_id', 'customer_id', 'order_date']]
df_order['order_date'] = pd.to_datetime(df_order['order_date'], format='%Y-%m-%d')
df_order.info()

<class 'pandas.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   order_id     646945 non-null  int64         
 1   customer_id  646945 non-null  int64         
 2   order_date   646945 non-null  datetime64[us]
dtypes: datetime64[us](1), int64(2)
memory usage: 14.8 MB


In [181]:
df_order = df_order[df_order.duplicated(subset=['customer_id'], keep=False)].sort_values(by=['customer_id', 'order_date', 'order_id'])
display(df_order)

,order_id,customer_id,order_date
4023,5280,1,2012-07-25
143252,184922,1,2014-05-31
238890,308113,1,2015-07-31
374571,483190,1,2017-04-23
544446,702081,1,2020-02-24
...,...,...,...
549990,709267,157563,2020-04-11
557438,718824,157563,2020-05-29
559725,721751,157563,2020-06-22
591841,763157,157563,2021-05-27


In [182]:

print(df_order['customer_id'].value_counts())

customer_id
139050    107
141899    105
141897    103
141898    100
139138     96
         ... 
157502      2
157503      2
157505      2
157530      2
157555      2
Name: count, Length: 67888, dtype: int64


In [183]:

df_order['inter_order_gap'] = df_order.groupby('customer_id')['order_date'].diff().dt.days
df_order['inter_order_gap'] = df_order['inter_order_gap'].fillna(0)
#df_order['inter_order_gap'] = df_order['inter_order_gap'].notna
print(df_order)

        order_id  customer_id order_date  inter_order_gap
4023        5280            1 2012-07-25              0.0
143252    184922            1 2014-05-31            675.0
238890    308113            1 2015-07-31            426.0
374571    483190            1 2017-04-23            632.0
544446    702081            1 2020-02-24           1037.0
...          ...          ...        ...              ...
549990    709267       157563 2020-04-11            225.0
557438    718824       157563 2020-05-29             48.0
559725    721751       157563 2020-06-22             24.0
591841    763157       157563 2021-05-27            339.0
637881    822601       157563 2022-08-31            461.0

[624587 rows x 4 columns]


In [184]:
df_order = df_order.sort_values(by=['customer_id', 'order_date'])
#df_check = df_order[df_order['inter_order_gap'] == 0]
df_filtered = df_order[df_order.groupby('customer_id').cumcount() > 0]
#print(df_check)
print(df_filtered)

        order_id  customer_id order_date  inter_order_gap
143252    184922            1 2014-05-31            675.0
238890    308113            1 2015-07-31            426.0
374571    483190            1 2017-04-23            632.0
544446    702081            1 2020-02-24           1037.0
586950    756884            1 2021-04-24            425.0
...          ...          ...        ...              ...
549990    709267       157563 2020-04-11            225.0
557438    718824       157563 2020-05-29             48.0
559725    721751       157563 2020-06-22             24.0
591841    763157       157563 2021-05-27            339.0
637881    822601       157563 2022-08-31            461.0

[556699 rows x 4 columns]


In [185]:
print(df_filtered.describe())

            order_id    customer_id                  order_date  \
count  556699.000000  556699.000000                      556699   
mean   450239.692944   86035.872175  2017-04-05 16:38:14.536903   
min       153.000000       1.000000         2012-07-04 00:00:00   
25%    259181.500000   41989.000000         2015-03-19 00:00:00   
50%    456552.000000   91618.000000         2016-12-30 00:00:00   
75%    647013.500000  134426.000000         2019-01-28 00:00:00   
max    834397.000000  157563.000000         2022-12-31 00:00:00   
std    227928.945563   48735.831389                         NaN   

       inter_order_gap  
count    556699.000000  
mean        285.592509  
min           0.000000  
25%          46.000000  
50%         144.000000  
75%         357.000000  
max        3785.000000  
std         389.691558  


Med = 144 -> C